Groundwater | Transport Modeling Track

# Step 8: Model Application — Verifying a Geothermal Doublet

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → **8-Apply** → 9-Document → 10-Communicate

**Story so far:** In [03t_modflow_transport.ipynb](03t_modflow_transport.ipynb) you met the MODFLOW 6 GWT transport packages and the advection–dispersion equation; in [04t_model_implementation.ipynb](04t_model_implementation.ipynb) you built and **locally refined** the source→receptor corridor and learned why a coarse grid (high grid Péclet) smears a plume. In [05t_calibration.ipynb](05t_calibration.ipynb) you constrained the transport parameters. This is the keystone: you now **use** that model to answer a real engineering question.

**The application — a geothermal doublet.** A *groundwater heat exchange* (**GWHE**) doublet pumps water from one well, passes it through a heat pump, and reinjects the thermally spent water at a second well a short distance away. A conservative tracer in the reinjected water lets us probe a central design concern: **how much of the reinjected water finds its way back to the extraction well?** Reinjected water that short-circuits to the extraction well degrades the thermal performance of the installation, so quantifying it matters.

| Section | Time budget |
|:--|:--|
| **Core — Rung 1 (scheme verification) + Rung 2 (doublet recirculation)** | ~20–30 min |
| *Optional — α_L sensitivity (Rung 2b) and reactive/contaminant extensions (Rungs 3–4)* | +20–40 min |

**Navigation:** ← Back to [05t_calibration.ipynb](05t_calibration.ipynb)  ·  → Forward to your **written report** (Steps 9–10 are the report + presentation; there are no 06t/07t notebooks in the transport track).

---

## §0 — Orientation: what are we trying to find out?

A geothermal doublet sits in the regional Limmat-Valley flow field. Reinjected water is partly swept downgradient and partly drawn back toward the pumping (extraction) well. Your job in Rung 2 is to **determine whether, and how much, the reinjected tracer short-circuits back to the extraction well** — not to assume the answer either way. The model gives you a number; you decide what it means.

### You **modify**, you do **not** build

The refined doublet model was built, refined, and validated for you in `transport_base_model.py` (the M2 base). In this notebook you *load* it and *read* results from it — and, optionally, change a single transport parameter (α_L) and rerun. **You never build or refine a grid here.** Grid construction lives in 04t.

### What this model CAN and CANNOT tell you

| The model gives a **defensible** answer for… | The model is **not trustworthy** for… |
|:--|:--|
| the **recirculation fraction** (steady-state plateau C∞/c_inj at the extraction well) | the **lateral width** of the contaminated/affected zone |
| **receptor breakthrough** — whether and when tracer arrives at the extraction well | the **exact position of a concentration contour** |
| the **longitudinal (centerline) reach** of the plume | anything governed by *transverse* dispersion |

The CANNOT column is dominated by **numerical-dispersion artefacts**: even on the refined grid the *transverse* grid Péclet stays Pe_T ≈ 12 ≫ 2 (the accuracy threshold you met in [02t](02t_perceptual_model.ipynb) / 04t), so lateral spreading is never fully resolved. This limitation is *taught, not fixed* — and it becomes a graded judgment in your report (see §5).

### Section index
- **§1** — Setup: load the pre-refined doublet, set toggles
- **Rung 1** — 1D scheme verification (provided black-box toy vs Ogata–Banks)  · *checkpoint 1*
- **Rung 2** — Doublet recirculation fraction  · *checkpoint 2*
- **Rung 2b** — α_L sensitivity (prediction in core; rerun toggled)  · *checkpoint 3*
- **Rungs 3–4** *(optional)* — riverbank solute source; reactive sorption + decay
- **§5** — Summary & report handoff

In [ ]:
# §1 — Setup
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Support modules + checkpoint utilities
sys.path.append(os.path.abspath('../../_SUPPORT/src'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts/scripts_exercises'))

import flopy
from tracer_test_utils import ogata_banks_btc, build_and_run_1d_verification
from transport_base_model import (
    load_doublet_base, build_doublet_base, LOCKED_PARAMS,
)
from shared_functions import create_multiple_choice

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# --- Toggles (long-running optional work is opt-in: default False) ---
FORCE_RERUN = False   # rebuild the doublet base instead of loading the cache
RUN_RUNG2B  = False   # rerun the doublet at 2x alpha_L (Rung 2b)
RUN_RUNG3   = False   # optional riverbank continuous solute source
RUN_RUNG4   = False   # optional reactive sorption + decay

# --- Locked transport parameters (see LOCKED_PARAMS) ---
print("Locked transport parameters:")
print(f"  alpha_L = {LOCKED_PARAMS['alh']} m   alpha_T = {LOCKED_PARAMS['ath1']} m")
print(f"  n_e     = {LOCKED_PARAMS['porosity']}   D_m* = {LOCKED_PARAMS['diffc']:.2e} m^2/d")

# --- Paths ---
DATA_DIR = os.path.expanduser('~/applied_groundwater_modelling_data/limmat')
MF6_EXE  = os.path.expanduser('~/.local/share/flopy/bin/mf6')
doublet_ws = os.path.join(DATA_DIR, 'transport_doublet_model', 'refined')


def _load_coarse():
    "Load the coarse NB4 flow model + GIS and locate the doublet (fallback only)."
    import geopandas as gpd
    csim = flopy.mf6.MFSimulation.load(
        sim_ws=os.path.join(DATA_DIR, 'notebook4_model'),
        exe_name=MF6_EXE, verbosity_level=0)
    cgwf = csim.get_model('limmat_valley')
    boundary = gpd.read_file(os.path.join(DATA_DIR, 'gis', 'limmat_model_boundary.gpkg'))
    rivers = gpd.read_file(os.path.join(DATA_DIR, 'gis', 'AV_Gewasser_-OGD.gpkg'))
    rivers = rivers[rivers['GEWAESSERNAME'].isin(['Limmat', 'Sihl'])
                    & rivers.intersects(boundary.geometry.iloc[0])]
    inj = (2679712.0, 1249990.0)
    spd = cgwf.output.budget().get_data(text='DATA-SPDIS')[0]
    mg = cgwf.modelgrid
    xc, yc = np.array(mg.xcellcenters), np.array(mg.ycellcenters)
    i0 = int(np.argmin((xc - inj[0]) ** 2 + (yc - inj[1]) ** 2))
    u = np.array([spd['qx'][i0], spd['qy'][i0]]); u = u / np.hypot(*u)
    ext = (inj[0] + 160 * u[0], inj[1] + 160 * u[1])
    return cgwf, boundary, rivers, inj, ext


# --- Load (or, if forced/missing, build) the pre-refined loc2 doublet ---
have_cache = os.path.exists(os.path.join(doublet_ws, 'base_cache.npz'))
if FORCE_RERUN or not have_cache:
    print("Building the refined doublet base (this is the heavy path)...")
    cgwf, boundary_gdf, river_gdf, inj_xy, ext_xy = _load_coarse()
    db = build_doublet_base(cgwf, boundary_gdf, river_gdf, inj_xy, ext_xy,
                            case_ws=doublet_ws, source_mode='well_aux',
                            Q=400.0, c_src=1.0, total_time=365.0)
else:
    db = load_doublet_base(doublet_ws)

print("\nLoaded refined doublet base:")
print(f"  cells (ncpl)      = {db.meta['ncpl']}")
print(f"  corridor Pe_L     = {db.meta['PeL_min']:.2f} .. {db.meta['PeL_max']:.2f}  (<= 2: longitudinal resolved)")
print(f"  Courant Cr peak   = {db.meta['Cr']:.2f}  (nstp = {db.meta['nstp']})")
print(f"  injection cell    = {db.src_cells[0]}   extraction cell = {db.receptor_cell}")

## Rung 1 — Does the transport scheme reproduce a known analytical breakthrough?

Before trusting *any* number the doublet model produces, we sanity-check the **GWT solute transport scheme itself** on a problem with an exact answer. This is a **provided black box**: `build_and_run_1d_verification()` builds a tiny 1D uniform-flow MODFLOW 6 GWF+GWT model with a first-type (constant-concentration) inlet, and we compare its breakthrough curve at a downstream observation point against the **two-term first-type Ogata–Banks (1961) analytical solution** for a continuous source. It is **decoupled from the doublet** — same advection–dispersion physics, but a clean 1D geometry.

**Vocabulary (continuous-source breakthrough).** A continuous source produces an S-shaped breakthrough curve C/C₀(t), not a pulse peak. We characterise it by:
- **toe** — the *first arrival*: the leading edge where C/C₀ first lifts off zero;
- **t₅₀** — the time at which C/C₀ = 0.5 (the breakthrough *midpoint*);
- **front-smearing** — how gradual the rise is. A perfectly resolved scheme gives a sharp, correctly-placed front; **numerical dispersion** (from a coarse grid) smears the front and shifts it.

Our pass/fail gate is therefore **front-sensitive**, not peak-based (a step BTC has no peak): we require ≤5% maximum-absolute C/C₀ error on the rising limb **and** ≤5% error in t₅₀.

In [ ]:
# Checkpoint 1 — committed BEFORE we run the verification toy.
create_multiple_choice('task_t08_checkpoint_1')

In [ ]:
# Rung 1 — run the provided 1D scheme-verification toy and compare to Ogata-Banks.
v_toy = 1.5                       # seepage velocity [m/d]
alpha_L_toy = LOCKED_PARAMS['alh']   # 10 m, the locked longitudinal dispersivity
n_e_toy = LOCKED_PARAMS['porosity']  # 0.20
D_m_toy = LOCKED_PARAMS['diffc']     # 8.64e-5 m^2/d
x_obs = 200.0                     # observation distance [m]
T_toy = 300.0                     # total time [d]

times1, c_num = build_and_run_1d_verification(
    v=v_toy, alpha_L=alpha_L_toy, n_e=n_e_toy, D_m=D_m_toy,
    x_obs=x_obs, total_time=T_toy)
meta1 = build_and_run_1d_verification.last_meta
x_ana = meta1['x_distance']       # exact analytic distance to the obs cell

# Analytical breakthrough with the SAME v and D_L = alpha_L*v + D_m.
c_ana = ogata_banks_btc(x_ana, times1, v_toy, alpha_L_toy, D_m=D_m_toy, c0=1.0)

# --- Front-sensitive error metrics ---
rising = c_ana > 0.01                       # rising limb (ignore the dead pre-toe tail)
front_err = float(np.max(np.abs(c_num[rising] - c_ana[rising])))
t50_num = float(np.interp(0.5, c_num, times1))
t50_ana = float(np.interp(0.5, c_ana, times1))
t50_rel = abs(t50_num - t50_ana) / t50_ana

print(f"grid Peclet  dx/alpha_L = {meta1['grid_Pe']:.2f}   Courant Cr = {meta1['Cr']:.2f}")
print(f"recovered v  = {meta1['v_sim']:.3f} m/d (target {v_toy})   D_L = {meta1['D_L']:.3f} m^2/d")
print(f"front max-abs C/C0 error = {100*front_err:.2f}%   (gate: <= 5%)")
print(f"t50 error                = {100*t50_rel:.2f}%   (num {t50_num:.1f} d vs ana {t50_ana:.1f} d; gate: <= 5%)")

fig, ax = plt.subplots()
ax.plot(times1, c_ana, 'k-', lw=2, label='Ogata-Banks (analytical)')
ax.plot(times1, c_num, 'o', ms=3, color='#1f77b4', alpha=0.6,
        label='MODFLOW 6 GWT (numerical)')
ax.axhline(0.5, color='gray', ls=':', lw=1)
ax.set_xlabel('Time [d]'); ax.set_ylabel('C / C$_0$ [-]')
ax.set_title(f'Rung 1 — 1D scheme verification at x = {x_ana:.0f} m '
             f'(front err {100*front_err:.1f}%, t50 err {100*t50_rel:.1f}%)')
ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout(); plt.show()

# Assert the front-sensitive gate (the scheme is verified if both pass).
assert front_err <= 0.05, f"Rising-limb C/C0 error {100*front_err:.2f}% exceeds 5%."
assert t50_rel <= 0.05, f"t50 error {100*t50_rel:.2f}% exceeds 5%."
print("\nScheme VERIFIED: numerical breakthrough matches Ogata-Banks to within the 5% front gate.")

**Rung 1 takeaway.** With Δx/α_L = 0.5 (well below the Pe ≤ 2 accuracy threshold) the MODFLOW 6 TVD advection + dispersion scheme reproduces the Ogata–Banks front to within a few percent in both rising-limb concentration and t₅₀. The transport *engine* is sound; any error in the doublet results below is about the **grid and geometry**, not the numerical method. This is exactly the point of checkpoint 1: a *failed* front gate would point to the grid (numerical dispersion from too-high a grid Péclet), not to a broken solver or a wrong analytical solution. ✅

## Rung 2 — How much of the reinjected tracer short-circuits to the extraction well?

Now the real question. The loaded model is the validated **loc2 geothermal doublet**: an injection well (reinjecting tracer at concentration c_inj = 1) and an extraction well 160 m away, both embedded in the regional Limmat flow field, on the locally-refined corridor. A conservative tracer is released continuously at the injection well; we monitor its breakthrough at the **extraction well**.

The decision-relevant quantity is the **recirculation fraction**:

$$\text{recirculation fraction} = \frac{C_\infty}{c_\text{inj}}$$

where C∞ is the **steady-state plateau** concentration at the extraction well (taken as the median of the last simulated samples). It is the fraction of the reinjected tracer concentration that finds its way back to the pumping well — a flux-integrated quantity that the grid converges on, hence defensible.

In [ ]:
# Checkpoint 2 — committed BEFORE we read the breakthrough result.
create_multiple_choice('task_t08_checkpoint_2')

In [ ]:
# Rung 2 — read the doublet breakthrough at the extraction well; compute recirculation fraction.
c_inj = 1.0
times2, bt = db.breakthrough            # times [d], C/c_inj at the extraction well
recirc = float(np.median(bt[-8:]) / c_inj)   # steady-state plateau ratio C_inf / c_inj
print(f"Steady-state plateau C_inf / c_inj = {recirc:.3f}")
print(f"  -> about {100*recirc:.0f}% of the reinjected tracer concentration "
      f"recirculates to the extraction well.")

# --- Breakthrough curve + plateau ---
fig, (axb, axm) = plt.subplots(1, 2, figsize=(14, 5))
axb.plot(times2, bt, '-', color='#d62728', lw=2)
axb.axhline(recirc, color='gray', ls='--', lw=1.5,
            label=f'plateau C$_\\infty$/c$_{{inj}}$ = {recirc:.2f}')
axb.set_xlabel('Time [d]'); axb.set_ylabel('C / c$_{inj}$ at extraction well [-]')
axb.set_title('Doublet breakthrough at the extraction well')
axb.set_ylim(0, 1); axb.legend(); axb.grid(True, alpha=0.3)

# --- Plume map at the final time ---
mg = db.modelgrid
cobj = db.gwt.output.concentration()
conc = np.clip(cobj.get_data(totim=times2[-1])[0, 0, :], 0, None)
pmv = flopy.plot.PlotMapView(modelgrid=mg, ax=axm)
ca = pmv.plot_array(conc, cmap='viridis', vmin=0, vmax=1)
sx, sy = db.src_xy; rx, ry = db.receptor_xy
axm.plot(sx, sy, 'v', color='white', mec='k', ms=12, label='injection (source)')
axm.plot(rx, ry, '^', color='red', mec='k', ms=12, label='extraction (receptor)')
axm.set_title(f'Tracer plume at t = {times2[-1]:.0f} d')
axm.set_xlabel('Easting [m]'); axm.set_ylabel('Northing [m]')
# Zoom to the doublet neighbourhood.
axm.set_xlim(min(sx, rx) - 250, max(sx, rx) + 250)
axm.set_ylim(min(sy, ry) - 250, max(sy, ry) + 250)
axm.legend(loc='upper right', fontsize=9)
fig.colorbar(ca, ax=axm, shrink=0.8, label='C / c$_{inj}$ [-]')
fig.tight_layout(); plt.show()

**Rung 2 result and judgment.** The plateau lands at **C∞/c_inj ≈ 0.51**: roughly **half** of the reinjected tracer short-circuits back to the extraction well. This is the *partial-recirculation* regime — neither full capture nor a clean fly-by. For a GWHE installation that is an operationally meaningful number: about half the thermally spent water is recycled.

**What is defensible, and what is not.** The recirculation fraction (and the receptor breakthrough generally) is a **flux-integrated** quantity that the grid converges on as you refine — it is the grid-robust, defensible claim. The **lateral width** of the affected zone is *not* trustworthy: the transverse grid Péclet stays Pe_T ≈ 12 ≫ 2 even on this refined grid, so transverse spreading is dominated by numerical dispersion. We do **not fix** this here — we *know* it and constrain our claims accordingly (carried into the report; see §5).

<details>
<summary>Optional reference — the analytical doublet arrival (Grove–Beetem)</summary>

For an idealised doublet of well spacing L, pumping rate Q, aquifer thickness b and porosity n_e in the *absence* of regional flow, the analytical first-arrival time along the connecting streamline is t ≈ π n_e b L² / (3 Q) (Grove & Beetem, 1971). It is shown here only as a conceptual anchor — our doublet sits in regional flow and is read numerically, so we do **not** grade against it.
</details>

## Rung 2b — Sensitivity: what if we doubled the longitudinal dispersivity?

α_L = 10 m is a calibrated best estimate, but it carries uncertainty. Before rerunning, **predict** how doubling it (α_L → 20 m) reshapes the extraction-well breakthrough. Remember the locked ratio keeps α_T = α_L/10, so **α_T also doubles** — and transverse spreading dilutes the corridor.

In [ ]:
# Checkpoint 3 — predict the effect of doubling alpha_L BEFORE the (toggled) rerun.
create_multiple_choice('task_t08_checkpoint_3')

In [ ]:
# Rung 2b — rerun the doublet at 2x alpha_L and overlay (cached; toggled off by default).
if RUN_RUNG2B:
    ws_2x = os.path.join(DATA_DIR, 'transport_doublet_model', 'refined_aL2x')
    if FORCE_RERUN or not os.path.exists(os.path.join(ws_2x, 'base_cache.npz')):
        cgwf, boundary_gdf, river_gdf, inj_xy, ext_xy = _load_coarse()
        # LOCKED_PARAMS is the module-level dict build_doublet_base reads; override + restore.
        _alh0, _ath0 = LOCKED_PARAMS['alh'], LOCKED_PARAMS['ath1']
        LOCKED_PARAMS['alh'] = 2.0 * _alh0      # alpha_L -> 20 m
        LOCKED_PARAMS['ath1'] = 2.0 * _ath0     # alpha_T also doubles
        try:
            db2 = build_doublet_base(cgwf, boundary_gdf, river_gdf, inj_xy, ext_xy,
                                     case_ws=ws_2x, source_mode='well_aux',
                                     Q=400.0, c_src=1.0, total_time=365.0)
        finally:
            LOCKED_PARAMS['alh'], LOCKED_PARAMS['ath1'] = _alh0, _ath0
    else:
        db2 = load_doublet_base(ws_2x)

    t2x, bt2x = db2.breakthrough
    recirc2x = float(np.median(bt2x[-8:]) / c_inj)
    fig, ax = plt.subplots()
    ax.plot(times2, bt, '-', color='#d62728', lw=2, label=f'alpha_L = 10 m (plateau {recirc:.2f})')
    ax.plot(t2x, bt2x, '-', color='#1f77b4', lw=2, label=f'alpha_L = 20 m (plateau {recirc2x:.2f})')
    ax.set_xlabel('Time [d]'); ax.set_ylabel('C / c$_{inj}$ at extraction well [-]')
    ax.set_title('Rung 2b — effect of doubling alpha_L'); ax.set_ylim(0, 1)
    ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout(); plt.show()
    print(f"alpha_L 10 -> 20 m: plateau {recirc:.2f} -> {recirc2x:.2f}")
else:
    print("RUN_RUNG2B = False. Set it to True to rerun the doublet at 2x alpha_L "
          "and confirm your checkpoint-3 prediction.")

**Rung 2b note.** Doubling α_L brings the tracer toe **earlier** (more leading-edge spreading) and broadens the rise with a **longer tail**. Because α_T doubles too, *transverse* dilution increases — so the steady-state plateau can **drop slightly** rather than rise. The plateau can never exceed c_inj (you cannot recover more than you inject). This is a prediction you record in the core; flip `RUN_RUNG2B = True` to confirm it.

## Rung 3 *(optional)* — A riverbank continuous solute source

The same base model accepts a *zero-water* contaminant source (constant-concentration cells) instead of the geothermal injection well. This optional rung releases a continuous solute source on a short line near the corridor and reads its breakthrough at the receptor — the contaminant-hydrogeology analogue of the doublet. **Optional and toggled off by default.**

In [ ]:
# Rung 3 (optional) — continuous CNC line source -> receptor breakthrough.
if RUN_RUNG3:
    ws3 = os.path.join(DATA_DIR, 'transport_doublet_model', 'refined_cnc')
    cgwf, boundary_gdf, river_gdf, inj_xy, ext_xy = _load_coarse()
    db3 = build_doublet_base(cgwf, boundary_gdf, river_gdf, inj_xy, ext_xy,
                             case_ws=ws3, source_mode='cnc', geometry='line',
                             geometry_size=20.0, c_src=1.0,
                             release={'type': 'continuous', 'duration_days': 365.0},
                             total_time=365.0)
    t3, bt3 = db3.breakthrough
    fig, ax = plt.subplots()
    ax.plot(t3, bt3, '-', color='#2ca02c', lw=2)
    ax.set_xlabel('Time [d]'); ax.set_ylabel('C / c$_{src}$ at receptor [-]')
    ax.set_title('Rung 3 — continuous solute line source breakthrough')
    ax.grid(True, alpha=0.3); fig.tight_layout(); plt.show()
    print(f"Receptor plateau C/c_src = {np.median(bt3[-8:]):.3f}")
else:
    print("RUN_RUNG3 = False (optional rung).")

## Rung 4 *(optional)* — A reactive (sorbing + decaying) solute

Conservative tracers ignore chemistry. This optional rung adds **linear sorption** (a distribution coefficient K_d, bulk density ρ_b) and **first-order decay** (λ) to the contaminant source — retardation delays arrival and decay lowers the plateau. **Optional and toggled off by default.**

In [ ]:
# Rung 4 (optional) — reactive sorption + first-order decay.
if RUN_RUNG4:
    ws4 = os.path.join(DATA_DIR, 'transport_doublet_model', 'refined_react')
    cgwf, boundary_gdf, river_gdf, inj_xy, ext_xy = _load_coarse()
    db4 = build_doublet_base(cgwf, boundary_gdf, river_gdf, inj_xy, ext_xy,
                             case_ws=ws4, source_mode='cnc', geometry='line',
                             geometry_size=20.0, c_src=1.0,
                             release={'type': 'continuous', 'duration_days': 365.0},
                             reactions={'Kd': 0.5, 'rho_b': 1800.0, 'lambda': 0.01},
                             total_time=365.0)
    t4, bt4 = db4.breakthrough
    fig, ax = plt.subplots()
    ax.plot(t4, bt4, '-', color='#9467bd', lw=2)
    ax.set_xlabel('Time [d]'); ax.set_ylabel('C / c$_{src}$ at receptor [-]')
    ax.set_title('Rung 4 — reactive (sorption + decay) breakthrough')
    ax.grid(True, alpha=0.3); fig.tight_layout(); plt.show()
    print(f"Reactive receptor plateau C/c_src = {np.median(bt4[-8:]):.3f} "
          f"(lower/later than conservative)")
else:
    print("RUN_RUNG4 = False (optional rung).")

## §5 — Summary & report handoff

**What you found.**
- **Rung 1** verified the GWT transport scheme: the 1D numerical breakthrough matches the Ogata–Banks analytical front to within the ≤5% front gate (rising-limb C/C₀ **and** t₅₀). The engine is trustworthy.
- **Rung 2** answered the application question: the recirculation fraction C∞/c_inj ≈ **0.51** — about half of the reinjected tracer short-circuits to the extraction well (partial-recirculation regime).
- **Rung 2b** framed the α_L sensitivity (earlier toe, broader front, longer tail; plateau may *drop* via transverse dilution and can never exceed c_inj).

**The defensible-threshold judgment is a GRADED report deliverable.** Restrict your reported claims to what the grid supports — **recirculation fraction / receptor arrival / longitudinal (centerline) reach** — and explicitly **warn against lateral contaminated-area or exact-contour claims** (transverse Pe_T ≫ 2 makes them numerical artefacts). This is the judgment node from the quality-judgment track. It is assessed under:
- `transport_track_milestones.md` **M4 AC4** (the deliverable must include a *defensible-threshold judgment* — longitudinal/receptor claims) and **AC5** (materials *warn against lateral contaminated-area threshold claims* and steer toward receptor arrival / well concentration / downgradient centerline statements);
- the **report rubric**: *Results & Interpretation* (20%, the limitations discussion) and *Conceptual Model* (15%, the assumptions you state and defend).

**Navigation:** ← [05t_calibration.ipynb](05t_calibration.ipynb)  ·  → your **written report** (Steps 9–10).

> *Section-completion note:* this notebook intentionally does **not** call `create_section_completion_marker` (that tracker is bound to 02t). Use the per-rung ✅ cues above to track your progress.